In [ ]:
import torch
import torch.nn as nn

# ResNet-18
Structure: Initial Input -> Stage 1 -> Stage 2 -> Stage 3 -> Stage 4 -> GAP -> FC
- WARNING: bias=False for all layers in ResNet-18, as mathematically BatchNorm cancelled out bias
- NOTE: ResNet-18 follows power of 2 down sampling: 224 -> 112 -> 56 -> 28 -> 14 -> 7
## Input
`B x C x H x W = B x 3 x 224 x 224`
## Initial Input Block
Conv2D(3, 64, 7, 2, 3) -> BatchNorm2D(64) -> ReLU -> MaxPooling(3, 2, 1)
- Input is B x 3 x 224 x 224
- Conv2D 7x7 kernel, 64 filters, stride 2, padding 3 (Parameters: 3 * 64 * 7 * 7)
- Input is reduced to B x 64 x 112 x 112
- BatchNorm 64 features (Parameters: 64 + 64)
- ReLU activation (Parameters: 0)
- Max Pooling 3x3 kernel, stride 2, padding 1 (Parameters: 0)
- Output is B x 64 x 56 x 56

In [ ]:
class InitialInput(nn.Module):
    def __init__(self, in_channels=3, out_channels=64):
        super(InitialInput, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.maxpool1(out)
        return out

## Basic Block - Residual Block
Conv2D(in_channels, out_channels, 3, stride, 1) -> BatchNorm2D(out_channels) -> ReLU -> Conv2D(out_channels, out_channels, 3, 1, 1) -> BatchNorm2D(out_channels) -> Skip Connection (Identical() or Conv2D(out_channels, out_channels,  1, 1, 0) -> BatchNorm(out_channels)) -> ReLU
- Input is B x in_channels x H x W
- Conv2D 3x3 kernel, out_channels filters, stride stride, padding 1 (Parameters: in_channels * out_channels * 3 * 3)
- Input is reduced to B x out_channels x H//2 x W//2 
- BatchNorm out_channels features (Parameters: out_channels + out_channels)
- ReLU activation (Parameters: 0)
- Conv2D 3x3 kernel, out_channels filters, stride 1, padding 1 (Parameters: out_channels * out_channels * 3 * 3)
- Input is kept at B x out_channels x H//2 x W//2 
- BatchNorm out_channels features (Parameters: out_channels + out_channels)
- Skip connection: Conv2D 1x1 kernel, out_channels filter, stride 2 (Parameters: out_channels * out_channels) then BatchNorm out_channels features (Parameters: out_channels + out_channels)
- ReLU activation
- IMPORTANT: First block has a stride of 1 for both, and in_channels = out_channels = 64, while others have 2 for first Conv2D and downscale by 2. This is due to image already shrinking to 25% when give to first block.


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Identity() if stride == 1 and in_channels == out_channels else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = torch.relu(out)
        return out

# Final Layer
GAP -> FC(num_classes)
- Global Average Pooling
- Fully Connected Layer 512 in_channels, num_classes out_channels (Parameters: 512 * num_classes)

In [ ]:
class FinalLayer(nn.Module):
    def __init__(self, in_channels, num_classes=1000):
        super(FinalLayer, self).__init__()
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(in_channels, num_classes)

    def forward(self, x):
        out = self.gap(x)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

# Parameters Calculation
Parameters (num_classes = 1000) =
+ (3 * 64 * 7 * 7) + (64 + 64) = 9,536
+ 2 * ((64 * 64 * 3 * 3) + (64 + 64) + (64 * 64 * 3 * 3) + (64 + 64)) = 147,968
+ ((64 * 128 * 3 * 3) + (128 + 128) + (128 * 128 * 3 * 3) + (128 + 128) + (64 * 128) + (128 + 128)) + ((128 * 128 * 3 * 3) + (128 + 128) + (128 * 128 * 3 * 3) + (128 + 128)) = 525,568
+ ((128 * 256 * 3 * 3) + (256 + 256) + (256 * 256 * 3 * 3) + (256 + 256) + (128 * 256) + (256 + 256)) + ((256 * 256 * 3 * 3) + (256 + 256) + (256 * 256 * 3 * 3) + (256 + 256)) = 2,099,712
+ ((256 * 512 * 3 * 3) + (512 + 512) + (512 * 512 * 3 * 3) + (512 + 512) + (256 * 512) + (512 + 512)) + ((512 * 512 * 3 * 3) + (512 + 512) + (512 * 512 * 3 * 3) + (512 + 512)) = 8,393,728
+ 512 * 1000 + 1000 = 513,000
= 11,689,512

Note: Don't forget bias for BatchNorm and FC just because all Conv2D ignore bias.


In [ ]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=1000):
        super(ResNet18, self).__init__()
        self.initial_input = InitialInput()
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        self.final_layer = FinalLayer(512, num_classes)

    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = []
        layers.append(BasicBlock(in_channels, out_channels, stride))
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.initial_input(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.final_layer(out)
        return out